# 12.4 多智能体 (Multi-Agent)

多智能体系统通过协作完成单个Agent难以完成的复杂任务。

本节涵盖：多智能体协作、角色扮演、辩论式、层级式

## 1. 多智能体协作与角色扮演

**协作模式**：多个Agent各司其职，通过消息传递协同工作。

**角色扮演**：每个Agent扮演特定角色（如产品经理、工程师、测试），按角色职责行动。

**消息传递**：Agent之间通过结构化消息通信，消息包含发送者、接收者、内容。

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

class Agent(nn.Module):
    def __init__(self, name, d=64, n_actions=4):
        super().__init__()
        self.name = name
        self.policy = nn.Sequential(nn.Linear(d, 32), nn.ReLU(), nn.Linear(32, n_actions))
        self.message_net = nn.Linear(d, d)

    def act(self, state):
        logits = self.policy(state)
        action = logits.argmax(dim=-1)
        message = self.message_net(state)
        return action, message

pm = Agent('Product_Manager')
eng = Agent('Engineer')
qa = Agent('QA_Tester')

state = torch.randn(1, 64)
print('=== Multi-Agent Collaboration ===')
print(f'\nRound 1: Product Manager defines requirements')
pm_action, pm_msg = pm.act(state)
print(f'  PM action: {pm_action.item()}, sends message to Engineer')

eng_state = state + 0.1 * pm_msg
eng_action, eng_msg = eng.act(eng_state)
print(f'\nRound 2: Engineer implements')
print(f'  Engineer action: {eng_action.item()}, sends message to QA')

qa_state = eng_state + 0.1 * eng_msg
qa_action, qa_msg = qa.act(qa_state)
print(f'\nRound 3: QA tests')
print(f'  QA action: {qa_action.item()}, sends feedback')
print(f'\nKey: Each agent has specialized role, messages carry information between agents.')

## 2. 辩论式与层级式

**辩论式**：多个Agent对同一问题提出不同观点，通过辩论达成共识或更优解。

**层级式**：Agent按层级组织，上层Agent负责规划和决策，下层Agent负责执行。

**适用场景**：
- 辩论式：需要多角度分析的问题
- 层级式：需要分解执行的复杂任务

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class DebateAgent(nn.Module):
    def __init__(self, d=64):
        super().__init__()
        self.argument_net = nn.Linear(d, d)
        self.stance_net = nn.Sequential(nn.Linear(d, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid())

    def argue(self, topic, prev_args=None):
        if prev_args is not None:
            topic = topic + 0.2 * prev_args.mean(dim=0, keepdim=True)
        argument = self.argument_net(topic)
        stance = self.stance_net(argument)
        return argument, stance

proponent = DebateAgent()
opponent = DebateAgent()

topic = torch.randn(1, 64)
print('=== Debate Mode ===')
all_args = []
for round_num in range(3):
    pro_arg, pro_stance = proponent.argue(topic, torch.cat(all_args) if all_args else None)
    all_args.append(pro_arg)
    opp_arg, opp_stance = opponent.argue(topic, torch.cat(all_args) if all_args else None)
    all_args.append(opp_arg)
    print(f'Round {round_num+1}: Pro={pro_stance.item():.3f}, Opp={opp_stance.item():.3f}')

print(f'\n=== Hierarchical Mode ===')
class HierarchicalAgent(nn.Module):
    def __init__(self, d=64, n_sub=3):
        super().__init__()
        self.manager = nn.Linear(d, n_sub)
        self.workers = nn.ModuleList([nn.Linear(d, d) for _ in range(n_sub)])
    def forward(self, x):
        assignment = self.manager(x).softmax(dim=-1)
        results = torch.stack([w(x) for w in self.workers])
        output = (assignment.T @ results.squeeze(1).unsqueeze(-1)).squeeze(-1)
        return output, assignment

hier = HierarchicalAgent()
x = torch.randn(1, 64)
output, assignment = hier(x)
print(f'Manager assigns: {assignment[0].detach().tolist()}')
print(f'Worker 0: {assignment[0, 0]:.2%}, Worker 1: {assignment[0, 1]:.2%}, Worker 2: {assignment[0, 2]:.2%}')
print(f'\nKey: Debate explores diverse perspectives, hierarchy decomposes complex tasks.')